In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'
models_dir = project_root / 'src' / 'models'

from src.models.model import PitStrategyNet

# Load data
X_test = np.load(processed_dir / 'X_test.npy')
y_test = np.load(processed_dir / 'y_test.npy')
meta = pd.read_csv(processed_dir / 'meta_test.csv')

with open(processed_dir / 'driver_mapping.json') as f:
    driver_mapping = {int(k): v for k, v in json.load(f).items()}

print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
# Load model
# Saving results
import yaml
from pathlib import Path

project_root = Path.cwd().resolve().parent.parent
with open(project_root / 'src' / 'config.yaml') as f:
    config = yaml.safe_load(f)

checkpoint = torch.load(models_dir / config['paths']['folder_name'] / config['paths']['model_filename'])
model = PitStrategyNet(
    input_dim    = checkpoint['input_dim'],
    hidden_dim_1 = config['model']['hidden_dim_1'],
    hidden_dim_2 = config['model']['hidden_dim_2'],
    dropout_rate = config['model']['dropout_rate']
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Model loaded.")

In [ ]:
# Standard evaluation
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

all_preds  = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(X_test_tensor), 32):
        batch = X_test_tensor[i:i+32]
        outputs = model(batch)
        preds = (torch.sigmoid(outputs) > 0.9878).float()
        all_preds.append(preds)
        all_labels.append(torch.tensor(y_test[i:i+32]))

all_preds = torch.cat(all_preds).squeeze()
all_labels = torch.cat(all_labels).squeeze()

print(classification_report(all_labels, all_preds, target_names=['No Pit', 'Pit']))
print(confusion_matrix(all_labels, all_preds))


In [ ]:
# MC Dropout uncertainty over full test set
model.train()  # dropout on, but we are not retraining the model. torch.no_grad() has frozen the weights
results = []

with torch.no_grad():
    for idx in range(len(X_test_tensor)):
        x = X_test_tensor[idx].unsqueeze(0)
        preds = torch.stack([torch.sigmoid(model(x)) for _ in range(100)])
        results.append({
            'driver': driver_mapping[int(meta.iloc[idx]['Driver'])],
            'lap':    int(meta.iloc[idx]['LapNumber']),
            'mean':   preds.mean().item(),
            'std':    preds.std().item(),
            'actual': int(y_test[idx][0])
        })

results_df = pd.DataFrame(results)
print(results_df.sort_values('std', ascending=False).head(20))

In [ ]:
# Saving the results to a .txt file
with open(project_root / 'src' / 'models' / config['paths']['folder_name'] / config['paths']['results_name'], "w") as c:
    c.write(classification_report(all_labels, all_preds, target_names=['No Pit', 'Pit']))